In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RetailSalesETL").getOrCreate()

df_raw = spark.read.csv(
    "/Workspace/Users/sauichyansalai@gmail.com/retail_sales",
    header=True,
    inferSchema=True
)

df_raw.show()

In [0]:
df_raw.printSchema()

In [0]:
df_raw.printSchema(10)
df_raw.count()

In [0]:
from pyspark.sql.functions import col, sum

df_raw.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
]).show()

In [0]:
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", "\t") \
    .csv("/Workspace/Users/sauichyansalai@gmail.com/retail_sales")

display(df_raw)

In [0]:
print(df_raw.columns)

In [0]:
print("Total rows:", df_raw.count())
print("Unique orders:", df_raw.select("order_id").distinct().count())

In [0]:
from pyspark.sql.functions import trim

df_silver = (
    df_raw
    .dropDuplicates(["order_id"])
    .dropna(subset=["order_id", "customer_id", "product"])
    .withColumn("product", trim(col("product")))
    .withColumn("category", trim(col("category")))
    .withColumn("city", trim(col("city")))
    .withColumn("payment_method", trim(col("payment_method")))
)

display(df_silver)

In [0]:
print("Silver records:", df_silver.count())

In [0]:
from pyspark.sql.functions import round

df_silver = df_silver.withColumn(
    "calculated_sales",
    round(
        col("quantity") *
        col("unit_price") *
        (1 - col("discount_pct")),
        2
    )
)

display(df_silver)

In [0]:
from pyspark.sql.functions import sum, avg, count

df_gold_category = (
    df_silver
    .groupBy("category")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("calculated_sales"), 2).alias("total_sales"),
        round(avg("calculated_sales"), 2).alias("average_order_value")
    )
    .orderBy(col("total_sales").desc())
)

display(df_gold_category)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("projects.project_workouts.retail_sales_silver")

In [0]:
df_gold_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("projects.project_workouts.retail_sales_gold")

In [0]:
%sql
SELECT
    ROUND(SUM(total_sales), 2) AS total_sales
FROM projects.project_workouts.retail_sales_gold;

In [0]:
%sql
SELECT
    category,
    total_orders,
    total_quantity,
    total_sales,
    average_order_value
FROM projects.project_workouts.retail_sales_gold
ORDER BY total_sales DESC;

In [0]:
%sql
SELECT
    product,
    COUNT(order_id) AS total_orders,
    SUM(quantity) AS total_quantity,
    ROUND(SUM(calculated_sales), 2) AS total_sales
FROM projects.project_workouts.retail_sales_silver
GROUP BY product
ORDER BY total_sales DESC
LIMIT 10;

In [0]:
%sql
SELECT
    city,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(calculated_sales), 2) AS total_sales
FROM projects.project_workouts.retail_sales_silver
GROUP BY city
ORDER BY total_sales DESC;

In [0]:
%sql
SELECT
    payment_method,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(calculated_sales), 2) AS total_sales
FROM projects.project_workouts.retail_sales_silver
GROUP BY payment_method
ORDER BY total_sales DESC;

                 retail_sales.csv
                       │
                       ▼
                 BRONZE / RAW
                       │
                       ▼
                Data Validation
                       │
                       ▼
                SILVER / CLEAN
                       │
                       ▼
              Transformations
                       │
                       ▼
                  GOLD DATA
                       │
                       ▼
                 SQL Analytics

In [0]:
spark.sql("SELECT * FROM projects.project_workouts.retail_sales_silver LIMIT 5").display()

In [0]:
spark.sql("select*from projects.project_workouts.retail_sales_gold limit 5").display()